# Corpus Statistics — Milestone 1

Data: Dummy corpus for validation purposes.
The figures below are not actual crawl results.
Analysis Units:
- Articles.
- Article versions.
- References, read once per article.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "corpus.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import pandas as pd
from IPython.display import display

from corpus import Corpus
from crawler.models import Status

DATA_DIR = ROOT / "data" / "test-corpus"

if not (DATA_DIR / "papers").is_dir():
    raise FileNotFoundError(f"The papers folder was not found in {DATA_DIR}")

corpus = Corpus(DATA_DIR)

In [ ]:
def has_text(value):
    return isinstance(value, str) and bool(value.strip())


valid_source_statuses = {
    Status.OK,
    Status.OK_NO_TEX,
    Status.NO_SOURCE,
}

paper_rows = []
version_rows = []

for paper in corpus.papers():
    versions = corpus.versions(paper.arxiv_id)

    if not versions:
        raise ValueError(f"Don't have version for paper {paper.arxiv_id}")

    expected = set(range(1, paper.num_versions + 1))
    observed = {version.version for version in versions}
    all_versions_recorded = observed == expected

    refs = corpus.references(paper.arxiv_id, versions[0].version)

    n_complete = sum(
        has_text(ref.ref_title)
        and any(has_text(author) for author in ref.ref_authors)
        and has_text(ref.ref_submission_date)
        for ref in refs
    )

    paper_rows.append(
        {
            "arxiv_id": paper.arxiv_id,
            "n_versions_expected": paper.num_versions,
            "n_versions_observed": len(versions),
            "all_versions_recorded": all_versions_recorded,
            "source_processed": all_versions_recorded
            and all(v.status in valid_source_statuses for v in versions),
            "has_tex": any(v.status == Status.OK for v in versions),
            "n_refs": len(refs),
            "n_refs_arxiv": sum(has_text(r.ref_arxiv_id) for r in refs),
            "refs_complete_ratio": n_complete / len(refs) if refs else None,
        }
    )

    for version in versions:
        version_rows.append(
            {
                "arxiv_id": paper.arxiv_id,
                "version": version.version,
                "status": version.status.value,
                "raw_archive_bytes": version.raw_archive_bytes,
                "raw_uncompressed_bytes": version.raw_uncompressed_bytes,
                "kept_bytes": version.kept_bytes,
            }
        )

if not paper_rows:
    raise ValueError("Papers not found")

papers_df = pd.DataFrame(paper_rows)
version_df = pd.DataFrame(version_rows)

display(papers_df)
display(version_df)

,arxiv_id,n_versions_expected,n_versions_observed,all_versions_recorded,source_processed,has_tex,n_refs,n_refs_arxiv,refs_complete_ratio
0,2408.11045,1,1,True,True,True,2,1,1.0


,arxiv_id,version,status,raw_archive_bytes,raw_uncompressed_bytes,kept_bytes
0,2408.11045,1,ok,0,889,889
